In [1]:
# Install required libraries (safe to run in Google Colab)
!pip -q install nltk pandas numpy scikit-learn

import re
import math
import random
from collections import Counter

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import brown, gutenberg, stopwords
from nltk.util import ngrams
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tag import hmm
from sklearn.datasets import fetch_20newsgroups

# Required NLTK resources
resources = [
    ("corpora/brown", "brown"),
    ("corpora/gutenberg", "gutenberg"),
    ("corpora/stopwords", "stopwords"),
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
    ("taggers/universal_tagset", "universal_tagset"),
]

for path, name in resources:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(name)

print("Setup complete.")


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...


Setup complete.


[nltk_data]   Unzipping taggers/universal_tagset.zip.


In [2]:
# Q1(a) Load the Brown Corpus and select the news category

brown_news_words = brown.words(categories="news")

print("Total raw tokens available in Brown news category:", len(brown_news_words))
print("First 30 raw tokens:")
print(brown_news_words[:30])


Total raw tokens available in Brown news category: 100554
First 30 raw tokens:
['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.', 'The', 'jury', 'further', 'said', 'in']


In [3]:
# Q1(b) Basic preprocessing
# - Convert to lowercase
# - Keep alphabetic word tokens
# - Use at least 5,000 tokens

processed_news = [
    word.lower()
    for word in brown_news_words
    if word.isalpha()
]

# The news category contains far more than 5,000 usable tokens.
news_tokens = processed_news[:5000]

print("Number of preprocessed tokens selected:", len(news_tokens))
print("First 50 preprocessed tokens:")
print(news_tokens[:50])


Number of preprocessed tokens selected: 5000
First 50 preprocessed tokens:
['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place', 'the', 'jury', 'further', 'said', 'in', 'presentments', 'that', 'the', 'city', 'executive', 'committee', 'which', 'had', 'charge', 'of', 'the', 'election', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'city', 'of', 'atlanta', 'for', 'the']


In [4]:
# Q1(c) Total tokens and vocabulary size

total_tokens = len(news_tokens)
vocabulary = set(news_tokens)
vocabulary_size = len(vocabulary)

q1_stats = pd.DataFrame({
    "Statistic": ["Total tokens", "Vocabulary size"],
    "Value": [total_tokens, vocabulary_size]
})

display(q1_stats)


,Statistic,Value
0,Total tokens,5000
1,Vocabulary size,1511


In [5]:
# Q1(d) Generate unigram, bigram, and trigram sequences

unigrams = list(ngrams(news_tokens, 1))
bigrams = list(ngrams(news_tokens, 2))
trigrams = list(ngrams(news_tokens, 3))

print("Number of unigram sequences:", len(unigrams))
print("Number of bigram sequences:", len(bigrams))
print("Number of trigram sequences:", len(trigrams))

print("\nSample unigrams:", unigrams[:10])
print("Sample bigrams:", bigrams[:10])
print("Sample trigrams:", trigrams[:10])


Number of unigram sequences: 5000
Number of bigram sequences: 4999
Number of trigram sequences: 4998

Sample unigrams: [('the',), ('fulton',), ('county',), ('grand',), ('jury',), ('said',), ('friday',), ('an',), ('investigation',), ('of',)]
Sample bigrams: [('the', 'fulton'), ('fulton', 'county'), ('county', 'grand'), ('grand', 'jury'), ('jury', 'said'), ('said', 'friday'), ('friday', 'an'), ('an', 'investigation'), ('investigation', 'of'), ('of', 'recent')]
Sample trigrams: [('the', 'fulton', 'county'), ('fulton', 'county', 'grand'), ('county', 'grand', 'jury'), ('grand', 'jury', 'said'), ('jury', 'said', 'friday'), ('said', 'friday', 'an'), ('friday', 'an', 'investigation'), ('an', 'investigation', 'of'), ('investigation', 'of', 'recent'), ('of', 'recent', 'primary')]


In [6]:
# Q1(e) Ten most frequent examples of each N-gram

unigram_freq = Counter(unigrams)
bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)

def frequency_table(counter, n=10):
    rows = []
    for gram, count in counter.most_common(n):
        rows.append({
            "N-gram": " ".join(gram),
            "Frequency": count
        })
    return pd.DataFrame(rows)

print("Top 10 Unigrams")
display(frequency_table(unigram_freq))

print("Top 10 Bigrams")
display(frequency_table(bigram_freq))

print("Top 10 Trigrams")
display(frequency_table(trigram_freq))


Top 10 Unigrams


,N-gram,Frequency
0,the,388
1,of,204
2,to,149
3,a,129
4,in,102
5,and,97
6,for,63
7,that,51
8,would,50
9,said,46


Top 10 Bigrams


,N-gram,Frequency
0,of the,64
1,in the,42
2,on the,16
3,the state,15
4,the jury,14
5,for the,14
6,would be,14
7,to the,13
8,that the,11
9,at the,10


Top 10 Trigrams


,N-gram,Frequency
0,the jury said,7
1,of the ward,6
2,the grand jury,4
3,is expected to,4
4,some of the,4
5,the precinct of,4
6,precinct of the,4
7,up to days,4
8,the fulton county,3
9,jury said it,3


In [7]:
# Q2(a) Load a suitable English text from Gutenberg

# Jane Austen's Emma is used as the English text.
gutenberg_words = gutenberg.words("austen-emma.txt")

print("Raw Gutenberg tokens:", len(gutenberg_words))
print("First 40 raw tokens:")
print(gutenberg_words[:40])


Raw Gutenberg tokens: 192427
First 40 raw tokens:
['[', 'Emma', 'by', 'Jane', 'Austen', '1816', ']', 'VOLUME', 'I', 'CHAPTER', 'I', 'Emma', 'Woodhouse', ',', 'handsome', ',', 'clever', ',', 'and', 'rich', ',', 'with', 'a', 'comfortable', 'home', 'and', 'happy', 'disposition', ',', 'seemed', 'to', 'unite', 'some', 'of', 'the', 'best', 'blessings', 'of', 'existence', ';']


In [8]:
# Q2(b) Tokenize and normalize the Gutenberg corpus
# We use alphabetic tokens, convert them to lowercase, and add sentence
# boundary markers so that the language model can learn sentence starts/ends.

raw_text = gutenberg.raw("austen-emma.txt")
sentences = sent_tokenize(raw_text)

normalized_sentences = []
for sentence in sentences:
    tokens = [
        word.lower()
        for word in word_tokenize(sentence)
        if word.isalpha()
    ]
    if tokens:
        normalized_sentences.append(tokens)

print("Number of non-empty sentences:", len(normalized_sentences))
print("Example normalized sentence:")
print(normalized_sentences[0][:30])


Number of non-empty sentences: 7459
Example normalized sentence:
['emma', 'by', 'jane', 'austen', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich', 'with', 'a', 'comfortable', 'home', 'and', 'happy', 'disposition', 'seemed', 'to', 'unite', 'some', 'of', 'the', 'best', 'blessings', 'of']


In [9]:
# Q2(c) Construct unigram and bigram frequency counts

# Add sentence boundary symbols.
START = "<s>"
END = "</s>"

unigram_counts = Counter()
bigram_counts = Counter()

for sent in normalized_sentences:
    padded = [START] + sent + [END]
    unigram_counts.update(padded)
    bigram_counts.update(ngrams(padded, 2))

vocab = set(unigram_counts.keys())
V = len(vocab)

print("Vocabulary size including boundary symbols:", V)
print("Total unigram count:", sum(unigram_counts.values()))
print("Total bigram count:", sum(bigram_counts.values()))

print("\nTop 10 unigrams:")
display(frequency_table(unigram_counts))

print("Top 10 bigrams:")
display(frequency_table(bigram_counts))


Vocabulary size including boundary symbols: 6934
Total unigram count: 172034
Total bigram count: 164575

Top 10 unigrams:


,N-gram,Frequency
0,< s >,7459
1,< / s >,7459
2,t h e,5201
3,t o,5181
4,a n d,4877
5,o f,4284
6,i,3177
7,a,3124
8,i t,2503
9,h e r,2448


Top 10 bigrams:


,N-gram,Frequency
0,<s> i,872
1,to be,605
2,of the,561
3,<s> she,501
4,in the,446
5,it was,446
6,<s> he,394
7,i am,394
8,<s> it,340
9,she had,332


In [10]:
# Q2(d) Add-one (Laplace) smoothing
#
# P(w_i | w_{i-1}) = (C(w_{i-1}, w_i) + 1) /
#                    (C(w_{i-1}) + V)

def smoothed_bigram_probability(previous_word, current_word):
    numerator = bigram_counts[(previous_word, current_word)] + 1
    denominator = unigram_counts[previous_word] + V
    return numerator / denominator

# Example probabilities
examples = [
    (START, "she"),
    ("she", "was"),
    ("this", "is"),
    ("computer", "science")
]

prob_rows = []
for prev_word, curr_word in examples:
    prob_rows.append({
        "Previous word": prev_word,
        "Current word": curr_word,
        "Smoothed probability": smoothed_bigram_probability(prev_word, curr_word)
    })

display(pd.DataFrame(prob_rows))


,Previous word,Current word,Smoothed probability
0,<s>,she,0.034878
1,she,was,0.035491
2,this,is,0.007641
3,computer,science,0.000144


In [11]:
# Q2(e) Score at least three test sentences
#
# The score used here is the log-probability of the sentence.
# Log probabilities avoid numerical underflow when multiplying many
# small probabilities.

def preprocess_test_sentence(sentence):
    tokens = [
        word.lower()
        for word in word_tokenize(sentence)
        if word.isalpha()
    ]
    return [START] + tokens + [END]

def sentence_log_probability(sentence):
    tokens = preprocess_test_sentence(sentence)
    log_prob = 0.0

    for prev_word, curr_word in ngrams(tokens, 2):
        p = smoothed_bigram_probability(prev_word, curr_word)
        log_prob += math.log(p)

    return log_prob

test_sentences = [
    "She was happy to see him.",
    "The little girl went to the house.",
    "The government announced a new policy."
]

score_rows = []
for sentence in test_sentences:
    score_rows.append({
        "Test sentence": sentence,
        "Log probability": sentence_log_probability(sentence)
    })

q2_scores = pd.DataFrame(score_rows)
display(q2_scores)


,Test sentence,Log probability
0,She was happy to see him.,-35.053433
1,The little girl went to the house.,-48.167522
2,The government announced a new policy.,-55.659601


In [12]:
# Q3(a) Load Brown Corpus with universal POS tags

tagged_sentences = brown.tagged_sents(
    categories="news",
    tagset="universal"
)

# Remove empty sentences, if any.
tagged_sentences = [sent for sent in tagged_sentences if sent]

print("Total tagged sentences:", len(tagged_sentences))
print("Example tagged sentence:")
print(tagged_sentences[0])


Total tagged sentences: 4623
Example tagged sentence:
[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


In [13]:
# Q3(b) Train/test split

random.seed(42)

# Shuffle sentence order reproducibly.
sentences_copy = tagged_sentences.copy()
random.shuffle(sentences_copy)

split_index = int(0.80 * len(sentences_copy))

train_sents = sentences_copy[:split_index]
test_sents = sentences_copy[split_index:]

print("Training sentences:", len(train_sents))
print("Testing sentences:", len(test_sents))
print("Training percentage:", round(100 * len(train_sents) / len(sentences_copy), 2), "%")
print("Testing percentage:", round(100 * len(test_sents) / len(sentences_copy), 2), "%")


Training sentences: 3698
Testing sentences: 925
Training percentage: 79.99 %
Testing percentage: 20.01 %


In [14]:
# Q3(c) Train an HMM-based POS tagger

trainer = hmm.HiddenMarkovModelTrainer()

# Training can take a little time because an HMM estimates
# initial, transition, and emission probabilities.
hmm_tagger = trainer.train(train_sents)

print("HMM POS tagger trained successfully.")


HMM POS tagger trained successfully.


In [15]:
# Q3(d) Predict tags for at least five unseen sentences

unseen_sentences = [
    "The company announced a new product",
    "The president visited the city",
    "A young student solved the problem",
    "The market showed strong growth",
    "Scientists developed a new method"
]

predictions = []

for sentence in unseen_sentences:
    tokens = sentence.lower().split()
    tagged = hmm_tagger.tag(tokens)
    predictions.append({
        "Sentence": sentence,
        "Predicted POS tags": " ".join(f"{word}/{tag}" for word, tag in tagged)
    })

display(pd.DataFrame(predictions))


/usr/local/lib/python3.13/dist-packages/nltk/tag/hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
/usr/local/lib/python3.13/dist-packages/nltk/tag/hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
/usr/local/lib/python3.13/dist-packages/nltk/tag/hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


,Sentence,Predicted POS tags
0,The company announced a new product,the/DET company/NOUN announced/VERB a/DET new/...
1,The president visited the city,the/DET president/NOUN visited/VERB the/DET ci...
2,A young student solved the problem,a/DET young/ADJ student/NOUN solved/VERB the/D...
3,The market showed strong growth,the/DET market/NOUN showed/VERB strong/ADJ gro...
4,Scientists developed a new method,scientists/NOUN developed/NOUN a/NOUN new/NOUN...


In [16]:
# Q3(e) Calculate test accuracy

# NLTK's HMM tagger provides an evaluate method for tagged test data.
test_accuracy = hmm_tagger.accuracy(test_sents)

print(f"HMM test accuracy: {test_accuracy * 100:.2f}%")


/usr/local/lib/python3.13/dist-packages/nltk/tag/hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


HMM test accuracy: 58.19%


In [17]:
# Q4(a)-(b) Ambiguity examples and contextual clues

ambiguity_examples = [
    {
        "Sentence": "I deposited money at the bank.",
        "Ambiguous word": "bank",
        "Intended meaning": "A financial institution",
        "Contextual clue": "deposited money"
    },
    {
        "Sentence": "The children sat on the bank of the river.",
        "Ambiguous word": "bank",
        "Intended meaning": "The land beside a river",
        "Contextual clue": "river"
    },
    {
        "Sentence": "The baseball player swung the bat.",
        "Ambiguous word": "bat",
        "Intended meaning": "A piece of sports equipment",
        "Contextual clue": "baseball player, swung"
    },
    {
        "Sentence": "A bat flew out of the cave at night.",
        "Ambiguous word": "bat",
        "Intended meaning": "A flying mammal",
        "Contextual clue": "flew, cave, night"
    },
    {
        "Sentence": "Please turn on the light.",
        "Ambiguous word": "light",
        "Intended meaning": "An illumination device/source",
        "Contextual clue": "turn on"
    },
    {
        "Sentence": "This bag is very light.",
        "Ambiguous word": "light",
        "Intended meaning": "Not heavy",
        "Contextual clue": "bag, weight"
    },
    {
        "Sentence": "She lit a match to start the fire.",
        "Ambiguous word": "match",
        "Intended meaning": "A small stick used to make fire",
        "Contextual clue": "lit, fire"
    },
    {
        "Sentence": "The blue shirt is a perfect match for his shoes.",
        "Ambiguous word": "match",
        "Intended meaning": "A suitable pairing",
        "Contextual clue": "shirt, shoes, perfect"
    }
]

display(pd.DataFrame(ambiguity_examples))


,Sentence,Ambiguous word,Intended meaning,Contextual clue
0,I deposited money at the bank.,bank,A financial institution,deposited money
1,The children sat on the bank of the river.,bank,The land beside a river,river
2,The baseball player swung the bat.,bat,A piece of sports equipment,"baseball player, swung"
3,A bat flew out of the cave at night.,bat,A flying mammal,"flew, cave, night"
4,Please turn on the light.,light,An illumination device/source,turn on
5,This bag is very light.,light,Not heavy,"bag, weight"
6,She lit a match to start the fire.,match,A small stick used to make fire,"lit, fire"
7,The blue shirt is a perfect match for his shoes.,match,A suitable pairing,"shirt, shoes, perfect"


In [18]:
# Q4(c) Coreference examples and antecedents

coreference_examples = [
    {
        "Sentence": "Riya submitted the assignment because she finished it early.",
        "Pronoun": "she",
        "Antecedent": "Riya",
        "Relation": "she refers to Riya"
    },
    {
        "Sentence": "The laptop was on the table, but it suddenly stopped working.",
        "Pronoun": "it",
        "Antecedent": "the laptop",
        "Relation": "it refers to the laptop"
    },
    {
        "Sentence": "The students entered the lab, and they started the experiment.",
        "Pronoun": "they",
        "Antecedent": "the students",
        "Relation": "they refers to the students"
    },
    {
        "Sentence": "Aman called Rahul after he reached home.",
        "Pronoun": "he",
        "Antecedent": "Aman/Rahul (ambiguous without more context)",
        "Relation": "the pronoun is not uniquely resolved by the sentence alone"
    },
    {
        "Sentence": "The company released a new phone, and its camera received praise.",
        "Pronoun": "its",
        "Antecedent": "the company",
        "Relation": "its refers to the company"
    }
]

display(pd.DataFrame(coreference_examples))


,Sentence,Pronoun,Antecedent,Relation
0,Riya submitted the assignment because she fini...,she,Riya,she refers to Riya
1,"The laptop was on the table, but it suddenly s...",it,the laptop,it refers to the laptop
2,"The students entered the lab, and they started...",they,the students,they refers to the students
3,Aman called Rahul after he reached home.,he,Aman/Rahul (ambiguous without more context),the pronoun is not uniquely resolved by the se...
4,"The company released a new phone, and its came...",its,the company,its refers to the company


In [19]:
# Q4(d) Load sci.space and comp.graphics from 20 Newsgroups

categories = ["sci.space", "comp.graphics"]

newsgroups = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    random_state=42
)

print("Categories:", newsgroups.target_names)
print("Number of documents:", len(newsgroups.data))
print("Documents per category:")
display(pd.Series(newsgroups.target, name="category").map(
    dict(enumerate(newsgroups.target_names))
).value_counts().rename_axis("Category").reset_index(name="Documents"))


Categories: ['comp.graphics', 'sci.space']
Number of documents: 1177
Documents per category:


,Category,Documents
0,sci.space,593
1,comp.graphics,584


In [20]:
# Q4(e) Extract at least 20 frequent domain-specific terms
#
# We remove stopwords and very short/non-alphabetic tokens, then count
# normalized terms within the selected two technical domains.

stop_words = set(stopwords.words("english"))

def clean_domain_text(text):
    tokens = re.findall(r"[A-Za-z]+", text.lower())
    return [
        token for token in tokens
        if len(token) >= 4 and token not in stop_words
    ]

all_domain_tokens = []
for document in newsgroups.data:
    all_domain_tokens.extend(clean_domain_text(document))

domain_term_counts = Counter(all_domain_tokens)

top_30_terms = pd.DataFrame(
    domain_term_counts.most_common(30),
    columns=["Domain term", "Frequency"]
)

print("Top 30 frequent terms across sci.space and comp.graphics:")
display(top_30_terms)


Top 30 frequent terms across sci.space and comp.graphics:


,Domain term,Frequency
0,space,1054
1,would,591
2,image,540
3,also,468
4,data,435
5,graphics,416
6,nasa,414
7,like,389
8,program,355
9,system,318


In [21]:
# Optional: compare the most frequent terms by domain

domain_tables = []

for category_index, category_name in enumerate(newsgroups.target_names):
    category_docs = [
        doc for doc, target in zip(newsgroups.data, newsgroups.target)
        if target == category_index
    ]

    category_tokens = []
    for document in category_docs:
        category_tokens.extend(clean_domain_text(document))

    counts = Counter(category_tokens)

    for term, freq in counts.most_common(15):
        domain_tables.append({
            "Category": category_name,
            "Term": term,
            "Frequency": freq
        })

display(pd.DataFrame(domain_tables))


,Category,Term,Frequency
0,comp.graphics,image,508
1,comp.graphics,graphics,411
2,comp.graphics,jpeg,271
3,comp.graphics,file,266
4,comp.graphics,also,232
5,comp.graphics,would,221
6,comp.graphics,data,219
7,comp.graphics,files,217
8,comp.graphics,software,213
9,comp.graphics,images,212
